# 00. Data Ingestion Sprint 1 
**Team B Traffic Volumes & Parking**

**Sprint 1 story:** *As a Data Engineer, I want to build a pipeline that aggregates event streams into
hourly average occupancy rates and traffic volumes so that data is summarized for performant serving.*

Before that aggregation can happen, this notebook fetches the raw datasets Team B evaluated
(see `Team B Datasets.xlsx` and `DATA.md`) directly from their public portals, and builds a
reusable **inner-Melbourne site filter** so the rest of the Sprint 1 EDA notebooks (`01`-`04`)
stay scoped to the study area (CBD + suburbs bordering it Yarra, Port Phillip, Stonnington,
Whitehorse, Maribyrnong, Melbourne) rather than all of statewide Victoria.

Datasets fetched here:
1. **SCATS Traffic Signal Volume Data** (opendata.transport.vic.gov.au) 15-min interval traffic counts, statewide, monthly ZIP.
2. **On-Street Parking Bay Sensors** (data.melbourne.vic.gov.au) live occupancy snapshot, City of Melbourne.
3. **Pop-up Bike Lanes** (opendata.transport.vic.gov.au) bike lane locations + trial dates, GeoJSON.
4. **Bicycle Infrastructure Network** (opendata.transport.vic.gov.au) existing cycling infrastructure, GeoJSON.
5. **Telemetry & TIRTL Traffic Counts** (opendata.transport.vic.gov.au) vehicle speed/classification counts, monthly ZIP.
6. **BoM weather (proxy)** the official BOM Climate Data Online portal blocks scripted downloads (HTTP 403).
   As a documented substitute we use the [Open-Meteo Historical Weather API](https://open-meteo.com/),
   a free, no-auth reanalysis dataset, for Melbourne CBD coordinates. This should be swapped for an
   authenticated BOM pull if the team gets portal access later.

All downloads are idempotent re-running this notebook skips files that already exist in `data/raw/`.


In [1]:
import os
import json
import zipfile
from pathlib import Path

import pandas as pd
import requests

RAW_DIR = Path("../../../data/raw")
PROCESSED_DIR = Path("../../../data/processed")
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


def download(url, dest, timeout=180):
    dest = Path(dest)
    if dest.exists():
        print(f"Already have {dest.name} ({dest.stat().st_size / 1e6:.1f} MB) - skipping.")
        return dest
    print(f"Downloading {dest.name} ...")
    resp = requests.get(url, timeout=timeout)
    resp.raise_for_status()
    dest.write_bytes(resp.content)
    print(f"Saved {dest.name} ({dest.stat().st_size / 1e6:.1f} MB)")
    return dest


def extract_zip(zip_path, dest_dir):
    dest_dir = Path(dest_dir)
    dest_dir.mkdir(exist_ok=True)
    if any(dest_dir.iterdir()):
        print(f"{dest_dir.name}/ already extracted - skipping.")
        return
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(dest_dir)
    print(f"Extracted {zip_path} -> {dest_dir}/")


## 1. SCATS Traffic Signal Volume Data + site list

In [2]:
SCATS_ZIP_URL = (
    "https://opendata.transport.vic.gov.au/dataset/331b846b-1e18-415f-a3f3-ce4198d86c82/"
    "resource/ce074c05-93ed-4ce8-bee4-c327527a743c/download/traffic_signal_volume_data_august_2026.zip"
)
SIGNALS_CSV_URL = (
    "https://opendata.transport.vic.gov.au/dataset/923af458-363d-469f-bc5e-84746a80b9a2/"
    "resource/1036a318-8867-4dc6-bd84-fc3bcd6cbdb9/download/victorian_traffic_signals.csv"
)

scats_zip = download(SCATS_ZIP_URL, RAW_DIR / "scats_august_2026.zip")
extract_zip(scats_zip, RAW_DIR / "scats_extracted")
download(SIGNALS_CSV_URL, RAW_DIR / "victorian_traffic_signals.csv")

scats_files = sorted((RAW_DIR / "scats_extracted").glob("*.csv"))
print(f"\n{len(scats_files)} daily SCATS files available:")
for f in scats_files:
    print(" ", f.name)


Already have scats_august_2026.zip (63.3 MB) - skipping.
scats_extracted/ already extracted - skipping.
Already have victorian_traffic_signals.csv (0.3 MB) - skipping.

14 daily SCATS files available:
  VSDATA_20260801.csv
  VSDATA_20260802.csv
  VSDATA_20260803.csv
  VSDATA_20260804.csv
  VSDATA_20260805.csv
  VSDATA_20260806.csv
  VSDATA_20260807.csv
  VSDATA_20260808.csv
  VSDATA_20260809.csv
  VSDATA_20260810.csv
  VSDATA_20260811.csv
  VSDATA_20260812.csv
  VSDATA_20260813.csv
  VSDATA_20260814.csv


## 2. On-Street Parking Bay Sensors (live occupancy snapshot)

In [3]:
PARKING_JSON_PATH = RAW_DIR / "on_street_parking_bay_sensors.json"

if PARKING_JSON_PATH.exists():
    print(f"Already have {PARKING_JSON_PATH.name} - skipping.")
else:
    print("Paginating City of Melbourne Opendatasoft records API...")
    all_records, limit, offset = [], 100, 0
    while True:
        r = requests.get(
            "https://data.melbourne.vic.gov.au/api/explore/v2.1/catalog/datasets/"
            "on-street-parking-bay-sensors/records",
            params={"limit": limit, "offset": offset}, timeout=30,
        )
        r.raise_for_status()
        data = r.json()
        if not data["results"]:
            break
        all_records.extend(data["results"])
        offset += limit
        if offset >= data["total_count"]:
            break
    PARKING_JSON_PATH.write_text(json.dumps(all_records))
    print(f"Fetched {len(all_records)} records.")

parking_records = json.loads(PARKING_JSON_PATH.read_text())
print(f"Total parking bay sensor records: {len(parking_records)}")


Already have on_street_parking_bay_sensors.json - skipping.
Total parking bay sensor records: 6324


## 3. Pop-up Bike Lanes + Bicycle Infrastructure Network

In [4]:
POPUP_URL = (
    "https://opendata.transport.vic.gov.au/dataset/2bedadc2-4e94-40f9-b754-0c394c6959f2/"
    "resource/2508cfe0-4694-45d0-b8bc-25be78e13c1a/download/popup_bike_lanes.geojson"
)
BIN_URL = (
    "https://opendata.transport.vic.gov.au/dataset/6cb739f0-ccf1-47a2-b215-192daf1e501a/"
    "resource/93e3301d-c559-44df-9435-01dfcad10794/download/bicycle_infrastructure_network.geojson"
)

download(POPUP_URL, RAW_DIR / "popup_bike_lanes.geojson")
download(BIN_URL, RAW_DIR / "bicycle_infrastructure_network.geojson", timeout=300)


Already have popup_bike_lanes.geojson (0.2 MB) - skipping.
Already have bicycle_infrastructure_network.geojson (32.5 MB) - skipping.


WindowsPath('../../../data/raw/bicycle_infrastructure_network.geojson')

## 4. Telemetry & TIRTL Traffic Counts + site lists

In [5]:
TIRTL_SITES_URL = (
    "https://opendata.transport.vic.gov.au/dataset/e2d78fb5-e16d-43b9-bcdc-607d9b4855f5/"
    "resource/1f685833-24fd-4eb0-af11-2e7cfc94da74/download/tirtl_sites.csv"
)
TIRTL_ZIP_URL = (
    "https://opendata.transport.vic.gov.au/dataset/e2d78fb5-e16d-43b9-bcdc-607d9b4855f5/"
    "resource/d0d451c2-e378-49c3-a1cd-0026b0f115a2/download/tirtl_15min_volume_classification_august_2026.zip"
)
TELEMETRY_SITES_URL = (
    "https://opendata.transport.vic.gov.au/dataset/4def599b-b545-4cde-a1aa-c7c3ece7e0b2/"
    "resource/f370b75d-06b3-46bf-ad9b-4dee7a7946cd/download/telemetry_sites.csv"
)
TELEMETRY_ZIP_URL = (
    "https://opendata.transport.vic.gov.au/dataset/4def599b-b545-4cde-a1aa-c7c3ece7e0b2/"
    "resource/9fa0ff57-116c-4c01-a8da-04c81567541b/download/telemetry_15min_volume_classification_august_2026.zip"
)

download(TIRTL_SITES_URL, RAW_DIR / "tirtl_sites.csv")
tirtl_zip = download(TIRTL_ZIP_URL, RAW_DIR / "tirtl_august_2026.zip", timeout=300)
extract_zip(tirtl_zip, RAW_DIR / "tirtl_extracted")

download(TELEMETRY_SITES_URL, RAW_DIR / "telemetry_sites.csv")
telemetry_zip = download(TELEMETRY_ZIP_URL, RAW_DIR / "telemetry_august_2026.zip")
extract_zip(telemetry_zip, RAW_DIR / "telemetry_extracted")


Already have tirtl_sites.csv (0.0 MB) - skipping.
Already have tirtl_august_2026.zip (40.5 MB) - skipping.
tirtl_extracted/ already extracted - skipping.
Already have telemetry_sites.csv (0.0 MB) - skipping.
Already have telemetry_august_2026.zip (0.9 MB) - skipping.
telemetry_extracted/ already extracted - skipping.


## 5. BoM weather (Open-Meteo proxy) for Melbourne CBD

In [6]:
WEATHER_PATH = RAW_DIR / "melbourne_cbd_weather_hourly.csv"

if WEATHER_PATH.exists():
    print(f"Already have {WEATHER_PATH.name} - skipping.")
else:
    params = {
        "latitude": -37.8136,
        "longitude": 144.9631,
        "start_date": "2026-07-01",
        "end_date": "2026-08-14",
        "hourly": "temperature_2m,precipitation,wind_speed_10m",
        "timezone": "Australia/Melbourne",
    }
    r = requests.get("https://archive-api.open-meteo.com/v1/archive", params=params, timeout=30)
    r.raise_for_status()
    weather_df = pd.DataFrame(r.json()["hourly"])
    weather_df.to_csv(WEATHER_PATH, index=False)
    print(f"Saved {len(weather_df)} hourly weather rows.")

weather_df = pd.read_csv(WEATHER_PATH)
weather_df.head()


Already have melbourne_cbd_weather_hourly.csv - skipping.

,time,temperature_2m,precipitation,wind_speed_10m
0,2026-07-01T00:00,11.9,0.0,13.6
1,2026-07-01T01:00,11.7,0.0,14.5
2,2026-07-01T02:00,11.4,0.0,14.0
3,2026-07-01T03:00,11.4,0.0,13.6
4,2026-07-01T04:00,11.9,0.0,13.4


## 6. Build the inner-Melbourne site filter

`SCATS`, `Telemetry` and `TIRTL` sites are statewide, but the study's priority sites are the
inner-Melbourne suburbs bordering the CBD (Prahran, Collingwood, North Melbourne, Burwood, Glen
Waverley per the PO meeting minutes) and the DATA.md target LGAs (Yarra, Port Phillip,
Merri-bek, Maribyrnong, Greater Dandenong, Greater Geelong).

The `victorian_traffic_signals.csv` site list has a `MUNICIPALITY` column, but it uses VicRoads'
internal abbreviation codes with no published crosswalk to LGA names. I decoded the ones relevant
to the study area by cross-referencing known intersections (e.g. `SWANSTON/LONSDALE` → `MBN` confirms
Melbourne; `HODDLE/JOHNSTON` → `YRA` confirms Yarra). Some codes (e.g. `MRD`) appear to mix multiple
LGAs inconsistently, so as a pragmatic Sprint 1 filter I combine the **confirmed codes** with a
**geographic bounding box** around inner Melbourne. This should be replaced with an exact join
against the official `street_spatial.gpkg` site list once the team receives it from Infrastructure
Victoria.


In [7]:
LAT_MIN, LAT_MAX = -37.87, -37.77
LON_MIN, LON_MAX = 144.88, 145.06
CONFIRMED_INNER_LGA_CODES = ["MBN", "YRA", "PTP", "STO", "WTH", "MRG"]  # Melbourne, Yarra, Port Phillip,
                                                                         # Stonnington, Whitehorse, Maribyrnong

def in_inner_melbourne(lat, lon):
    return lat.between(LAT_MIN, LAT_MAX) & lon.between(LON_MIN, LON_MAX)

# --- SCATS ---
signals = pd.read_csv(RAW_DIR / "victorian_traffic_signals.csv")
scats_inner = signals[
    signals["MUNICIPALITY"].isin(CONFIRMED_INNER_LGA_CODES) | in_inner_melbourne(signals["LATITUDE"], signals["LONGITUDE"])
].copy()
scats_inner.to_csv(PROCESSED_DIR / "scats_inner_melbourne_sites.csv", index=False)
print(f"SCATS: {len(scats_inner)} / {len(signals)} sites fall within the inner-Melbourne study scope.")

# --- Telemetry ---
telemetry_sites = pd.read_csv(RAW_DIR / "telemetry_sites.csv")
telemetry_inner = telemetry_sites[
    in_inner_melbourne(telemetry_sites["latitude"], telemetry_sites["longitude"])
].copy()
telemetry_inner.to_csv(PROCESSED_DIR / "telemetry_inner_melbourne_sites.csv", index=False)
print(f"Telemetry: {len(telemetry_inner)} / {len(telemetry_sites)} sites fall within the inner-Melbourne study scope.")

# --- TIRTL ---
tirtl_sites = pd.read_csv(RAW_DIR / "tirtl_sites.csv")
tirtl_inner = tirtl_sites[
    in_inner_melbourne(tirtl_sites["latitude"], tirtl_sites["longitude"])
].copy()
tirtl_inner.to_csv(PROCESSED_DIR / "tirtl_inner_melbourne_sites.csv", index=False)
print(f"TIRTL: {len(tirtl_inner)} / {len(tirtl_sites)} sites fall within the inner-Melbourne study scope.")


SCATS: 1325 / 5066 sites fall within the inner-Melbourne study scope.
Telemetry: 0 / 55 sites fall within the inner-Melbourne study scope.
TIRTL: 10 / 406 sites fall within the inner-Melbourne study scope.


In [8]:
import numpy as np

cbd_lat, cbd_lon = -37.8136, 144.9631
telemetry_sites["dist_km_from_cbd"] = np.sqrt(
    (telemetry_sites["latitude"] - cbd_lat) ** 2 + (telemetry_sites["longitude"] - cbd_lon) ** 2
) * 111

print("Telemetry has 0 inner-Melbourne sites - it's a strategic/freeway monitoring network, not an")
print("inner-city one. Nearest telemetry sites to the CBD:\n")
print(telemetry_sites.sort_values("dist_km_from_cbd")[["site", "site_description", "dist_km_from_cbd"]].head(10).to_string(index=False))


Telemetry has 0 inner-Melbourne sites - it's a strategic/freeway monitoring network, not an
inner-city one. Nearest telemetry sites to the CBD:

 site                                      site_description  dist_km_from_cbd
 1137         Fitzsimons Ln Btwn Summerhill Rd & Kiwanis Ln         20.742988
10613            Hume Hwy South of Beveridge At 36.9Km Post         35.914399
10005             Maroondah Hwy At Coldstream @ 41.1Km Post         46.214010
12831                 Calder Fwy 400M South of Blackwood Rd         61.881293
 1161 Princes Fwy West Southwest of Avalon Rd Lara @ 59.5Km         65.511626
10001             Hume Fwy South of Broadford @ 70.0Km Post         66.517126
10004             Melba Hwy North of Glenburn @ 82.2Km Post         70.996047
10011           Northern Hwy South of Pyalong @ 76.1Km Post         72.283120
 1481                     Bass Hwy 500m South of Jetty Lane         84.854941
10009              South of Gheringhap  (50m S of Ryans Rd)         86.0612

### Findings
- SCATS has **1,325 of 5,066** statewide sites (~26%) within the inner-Melbourne study scope a
  strong signal source for the core stream.
- TIRTL has **10 of 406** sites nearby (mostly freeway/arterial speed-camera-style sensors), useful as
  supplementary context but sparse for street-level analysis.
- **Telemetry has zero sites within inner Melbourne** the nearest is ~21km away on Fitzsimons Lane.
  This dataset is a statewide/regional highway monitoring network and is a poor fit for the
  CBD-adjacent priority sites; it may be more useful for the regional bands of the ~33 intervention
  sites (e.g. Greater Geelong, Greater Dandenong corridors) once those exact sites are confirmed.
- All outputs are saved to `data/processed/*_inner_melbourne_sites.csv` for reuse by notebooks `01`-`04`.
